# Steam EDA: what price will a user review next?

Target: the list price of the next game a user reviews, predicted from their history alone.
Needs `data/cache/reviews.parquet` (built by `steam_price_eda_karina.ipynb`).

In [1]:
from pathlib import Path

import numpy as np
import polars as pl
import hvplot.polars  # noqa: F401
import plotly.express as px
import plotly.io as pio

from utils import LABELS, UserTimeline, bucket, load_games

pio.renderers.default = "plotly_mimetype+notebook_connected"  # also shows in the HTML export
pl.Config.set_tbl_rows(20)
DATA = Path("data")

## Load

`australian_users_items.json` isn't used: only 6,187 of the 1.49M users with a `user_id` (0.4%) appear in it.

In [2]:
games = load_games(DATA / "steam_games.json")
rev = pl.read_parquet(DATA / "cache" / "reviews.parquet").unique()
ev = rev.join(games.drop_nulls("price").select("product_id", "price", "genres", "title"), on="product_id")

print(f"games {games.height:,} | reviews {rev.height:,} (deduplicated) | labelled {ev.height:,} "
      f"| with user_id {ev['user_id'].is_not_null().sum():,}")

games 32,132 | reviews 6,961,672 (deduplicated) | labelled 6,893,637 | with user_id 2,806,060


In [3]:
outcome = (pl.when(pl.col("price_raw").is_null()).then(pl.lit("missing"))
             .when(pl.col("price").is_null()).then(pl.lit("dropped"))
             .when(pl.col("price") == 0).then(pl.lit("free"))
             .otherwise(pl.lit("paid")))
(games.group_by(outcome.alias("label"), pl.when(outcome == "dropped").then("price_raw").alias("raw value"))
      .len().sort("len", descending=True))

label,raw value,len
str,str,u32
"""paid""",null,28845
"""free""",null,1900
"""missing""",null,1377
"""dropped""","""Play Now""",2
"""dropped""","""Third-party""",2
"""dropped""","""Play WARMACHINE: Tactics Demo""",1
"""dropped""","""Install Theme""",1
"""dropped""","""Starting at $449.00""",1
"""dropped""","""Starting at $499.00""",1


In [4]:
coverage = (rev.join(games.select("product_id", "price"), on="product_id", how="left")
              .sort("date").group_by_dynamic("date", every="3mo")
              .agg(labelled=pl.col("price").is_not_null().mean())
              .filter(pl.col("date") >= pl.date(2012, 1, 1)))
print(f"review events with a price: {ev.height / rev.height:.1%}")
coverage.hvplot.line("date", "labelled", ylim=(0, 1.02), width=750, height=220,
                     ylabel="share with a price", title="Label coverage by quarter")

review events with a price: 99.0%


:Curve   [date]   (labelled)

## The target: price of a reviewed game

Review-weighted, not catalogue-weighted: popular games count many times.

In [5]:
points = (ev.group_by("price").len().sort("len", descending=True).head(20)
            .with_columns(label=pl.format("${}", pl.col("price"))))
points.hvplot.bar("label", "len", rot=45, width=750, height=300,
                  xlabel="list price", ylabel="reviews", yformatter="%.0f", title="Top 20 price points by review volume")

:Bars   [label]   (len)

In [6]:
lp = lambda df, name: df.select(pl.col("price").log1p().alias("log1p price")).hvplot.hist(
    bins=70, bin_range=(0, 7), normed=True, alpha=0.5, label=name)
(lp(games.drop_nulls("price"), "catalogue") * lp(ev, "review events")).opts(
    width=750, height=300, title="Catalogue vs review-weighted price (log1p)", legend_position="top_right")

:Overlay
   .Histogram.I  :Histogram   [log1p price]   (Frequency)
   .Histogram.II :Histogram   [log1p price]   (Frequency)

## Time

There's no geography in the data, so the closest thing to a map is the calendar: Steam's summer and winter sales show up as hot weeks.

In [7]:
daily = (rev.group_by("date").len()
           .with_columns(year=pl.col("date").dt.year(), week=pl.col("date").dt.week())
           .filter(pl.col("year") >= 2012))
daily.hvplot.heatmap("week", "year", "len", reduce_function=np.sum, logz=True, cmap="viridis",
                     width=850, height=300, clabel="reviews (log)", title="Reviews per week")

:HeatMap   [week,year]   (len)

In [8]:
weekly_price = (ev.with_columns(year=pl.col("date").dt.year(), week=pl.col("date").dt.week())
                  .filter(pl.col("year") >= 2012)
                  .group_by("year", "week").agg(pl.col("price").mean()))
weekly_price.hvplot.heatmap("week", "year", "price", cmap="magma", width=850, height=300,
                            clabel="mean $", title="Mean price of reviewed games, by week")

:HeatMap   [week,year]   (price)

In [9]:
monthly = (ev.sort("date").group_by_dynamic("date", every="1mo")
             .agg(mean_price=pl.col("price").mean(), free_share=(pl.col("price") == 0).mean(), reviews=pl.len())
             .filter(pl.col("date") >= pl.date(2012, 1, 1)))
(monthly.hvplot.line("date", "mean_price", ylabel="mean $", height=220, width=750)
 + monthly.hvplot.line("date", "free_share", ylabel="share free", height=220, width=750)
 + monthly.hvplot.area("date", "reviews", yformatter="%.0f", height=220, width=750)).cols(1)

:Layout
   .Curve.I  :Curve   [date]   (mean_price)
   .Curve.II :Curve   [date]   (free_share)
   .Area.I   :Area   [date]   (reviews)

## Genre × price

Row-normalised: within each genre, where do reviewed games sit on the price scale?

In [10]:
top_genres = ev.explode("genres").group_by("genres").len().sort("len", descending=True).head(15)["genres"]
genre_price = (ev.explode("genres").filter(pl.col("genres").is_in(top_genres.to_list()))
                 .with_columns(bucket("price"))
                 .group_by("genres", "price_b").len()
                 .with_columns(share=pl.col("len") / pl.col("len").sum().over("genres")))
genre_price.hvplot.heatmap("price_b", "genres", "share", cmap="blues", width=750, height=450,
                           xticks=list(enumerate(LABELS)), xlabel="price", ylabel="", title="Price mix by genre")

:HeatMap   [price_b,genres]   (share)

## Users: is next price predictable from history?

`user_id` is only present on ~40% of reviews, so sequences use that subset.

In [11]:
seq = (ev.drop_nulls("user_id").sort("user_id", "date", "product_id")
         .with_columns(k=pl.int_range(pl.len()).over("user_id"),
                       prev=pl.col("price").shift().over("user_id"),
                       prev_date=pl.col("date").shift().over("user_id"),
                       csum=pl.col("price").cum_sum().over("user_id"))
         .with_columns(hist_mean=(pl.col("csum") - pl.col("price")) / pl.col("k")))
pairs = seq.filter(pl.col("k") > 0)
same_day = (pairs["date"] == pairs["prev_date"]).mean()
print(f"{pairs.height:,} (history → next) pairs from {pairs['user_id'].n_unique():,} users")
print(f"{same_day:.1%} of next reviews share a date with the previous one (ordered by product_id)")

1,330,464 (history → next) pairs from 445,864 users
16.5% of next reviews share a date with the previous one (ordered by product_id)


In [12]:
transitions = (pairs.with_columns(bucket("prev"), bucket("price"))
                   .group_by("prev_b", "price_b").len()
                   .with_columns(share=pl.col("len") / pl.col("len").sum().over("prev_b")))
transitions.hvplot.heatmap("price_b", "prev_b", "share", cmap="blues", width=600, height=480,
                           xticks=list(enumerate(LABELS)), yticks=list(enumerate(LABELS)),
                           xlabel="next price", ylabel="previous price", title="P(next bucket | previous bucket)")

:HeatMap   [price_b,prev_b]   (share)

In [13]:
global_mean = pairs["price"].mean()
by_k = (pairs.with_columns(seen=pl.col("k").clip(upper_bound=30))
              .group_by("seen").agg(
                  last_price=(pl.col("price") - pl.col("prev")).abs().mean(),
                  history_mean=(pl.col("price") - pl.col("hist_mean")).abs().mean(),
                  global_mean=(pl.col("price") - global_mean).abs().mean())
              .sort("seen"))
by_k.hvplot.line("seen", ["last_price", "history_mean", "global_mean"], width=750, height=320,
                 xlabel="reviews already seen (capped at 30)", ylabel="MAE ($)",
                 title="Naive predictors vs history length")

:NdOverlay   [Variable]
   :Curve   [seen]   (value)

## Temporal split

Train before T (80% of reviews), validate to T' (90%), test after. Cold start: a val/test user with no reviews before T.

In [14]:
dates = seq["date"].sort()
T, T2 = dates[int(0.8 * len(dates))], dates[int(0.9 * len(dates))]
seen = seq.filter(pl.col("date") < T)["user_id"].unique()
split = (pl.when(pl.col("date") < T).then(pl.lit("train"))
           .when(pl.col("date") < T2).then(pl.lit("val")).otherwise(pl.lit("test")))
print(f"T = {T}   T' = {T2}")
(pairs.with_columns(split=split, cold=~pl.col("user_id").is_in(seen.implode()))
      .group_by("split")
      .agg(pairs=pl.len(), mean_price=pl.col("price").mean(),
           free_share=(pl.col("price") == 0).mean(), cold_start=pl.col("cold").mean())
      .sort(pl.col("split").replace_strict({"train": 0, "val": 1, "test": 2})))

T = 2017-07-07 00:00:00   T' = 2017-11-06 00:00:00


split,pairs,mean_price,free_share,cold_start
str,u32,f64,f64,f64
"""train""",1023986,16.054699,0.162581,0.0
"""val""",156616,18.748116,0.174867,0.135842
"""test""",149862,19.380302,0.142911,0.22508


### User price personas

One point per user with ≥5 reviews (sample of 15k). Drag to rotate.

In [15]:
personas = (seq.group_by("user_id")
               .agg(n=pl.len(), mean_price=pl.col("price").mean(), std_price=pl.col("price").std(),
                    free_share=(pl.col("price") == 0).mean())
               .filter(pl.col("n") >= 5).sample(15_000, seed=0))
px.scatter_3d(personas.to_pandas(), x="mean_price", y="std_price", z="n", color="free_share",
              log_z=True, opacity=0.6, height=600, color_continuous_scale="Viridis",
              labels={"n": "reviews", "mean_price": "mean $", "std_price": "std $"}).update_traces(marker_size=2)

### Explore single users

Each dot is a reviewed price, in order (hover for the game). The red dot is the last review, the one a model would predict.

In [16]:
timelines = (seq.group_by("user_id")
                .agg(d=pl.col("date").dt.strftime("%Y-%m-%d"), t="title", p="price")
                .filter(pl.col("p").list.len().is_between(8, 40))
                .sample(300, seed=1))
timelines.write_json("demos/user_timelines.json")  # the book version below reads this file
UserTimeline(users=timelines.to_dicts())

```{anywidget} demos/user_timeline.js
{"url": "user_timelines.json"}
```